# PEKA reproduction — Kaggle T4 x2

Repo: `duyh80456-code/peka-rebuild` (public, track A / breast only)

**Muc tieu:** baseline H-optimus-0 **~0.624**  ->  PEKA **~0.698**

## Cach dung
Doi DUY NHAT `RUN_PHASE` o cell duoi, roi **Run All**.
Xong thi **Save Version**, va attach Output do lam **Input** cho session sau.

| PHASE | Lam gi | Thoi gian | Can GPU? |
|---|---|---|---|
| 1 | download HEST + cat patch + align gene | 4-6h | **khong** |
| 2 | scFoundation embedding | 3-5h | co |
| 3 | kmeans + smoke + train — **lap lai den khi xong** | 12h/session | co |
| 4 | extract feature + HVG + eval -> **PCC** | ~1h | co |
| 5 | baseline image_encoder (doi chung) | ~1h | co |

> PHASE 1 khong dung GPU. Chay no o session **Accelerator: None** de tiet kiem
> quota GPU (~30h/tuan) cho cac phase sau.

## Truoc khi chay
1. Bat **Internet** (va **GPU T4 x2** tu PHASE 2 tro di)
2. Tao Kaggle Secret **`HF_TOKEN`**
3. Vao HuggingFace **chap nhan dieu khoan** (khong lam se loi 403 giua chung):
   - huggingface.co/datasets/MahmoodLab/hest
   - huggingface.co/bioptimus/H-optimus-0

## Attach Input nao?
Attach Output cua lan chay **thanh cong** gan nhat, KHONG phai lan moi nhat.
Mot lan chay chet giua chung (het dia, het gio) van tao ra Output, nhung file
trong do co the **cut** ma van dem du 8 -> bang ton kho bao OK trong khi du
lieu hong. Cell setup co buoc `[integrity]` mo thu tung file de bat viec nay;
neu no bao HONG thi doi sang version cu hon.

## Train nhieu session
PHASE 3 tu tim `last.ckpt` va truyen `--resume`. Cu chay lai PHASE 3
cho den khi log bao du `EPOCHS`. State optimizer/scheduler/epoch duoc khoi phuc day du.


In [ ]:
# ================= CHI DOI O DAY =================
RUN_PHASE  = 7        # 1..7  (7 = chay code GOC cua paper, khong can GPU)
EPOCHS     = 50       # paper: 50
BATCH_SIZE = 8        # giam ve 4 neu OOM
ACCUM      = 4        # batch hieu dung = BATCH_SIZE * ACCUM = 32 (paper)
PEFT       = "bone"   # phuong phap cua paper (Block-Affine)
ENCODER    = "H-optimus-0"
# Dung co trat tu truoc khi Kaggle giet kernel o 12h. Kernel bi giet thi khoi
# finally KHONG chay -> last.ckpt ket lai o /kaggle/tmp (khong duoc luu) va ca
# session coi nhu mat trang. 10h45 chua ~1h du de ghi + commit Output.
MAX_TIME   = "00:09:45:00"
EVAL_CKPT  = "last"   # "last" = epoch moi nhat | "best" = val_kd_loss thap nhat
# val_kd_loss TANG deu tu epoch 7 nhung PCC do duoc lai TANG theo (0.1096 o
# epoch 12 -> 0.1256 o epoch 14). Nghia la "best" theo val_kd_loss dang chi ve
# epoch 7, gan nhu chac chan te hon epoch moi nhat. Mac dinh doi sang "last".
# =================================================
print("RUN_PHASE =", RUN_PHASE)

In [ ]:
import os, subprocess, sys, glob
from pathlib import Path

REPO_URL = "https://github.com/duyh80456-code/peka-rebuild.git"
REPO     = Path("/kaggle/working/peka-rebuild")
SCRATCH  = Path("/kaggle/tmp/peka")        # KHONG duoc luu -> de HEST tho o day
OUT      = Path("/kaggle/working/OUTPUT")  # duoc luu thanh Output
EXP      = ENCODER + "_" + PEFT + "_breast_in_hest_joint"
SCRATCH.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

import shutil
for _p in ("/kaggle/working", "/kaggle/tmp"):
    _u = shutil.disk_usage(_p)
    print("%-16s trong %5.1f GiB / %5.1f GiB" % (_p, _u.free / 1024**3, _u.total / 1024**3))

def run(cmd, env=None, cwd=REPO):
    print("+", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = (UserSecretsClient().get_secret("HF_TOKEN") or "").strip()
assert os.environ["HF_TOKEN"], "Thieu Kaggle Secret HF_TOKEN"

if not (REPO / ".git").is_dir():
    run(["git", "clone", "--recursive", REPO_URL, str(REPO)], cwd="/kaggle/working")
else:
    run(["git", "pull", "--ff-only"])

# QUAN TRONG: --no-deps. Kaggle da co san torch/lightning/scanpy/hydra_zen...
# Neu de pip giai deps cua peka (numpy, pandas... deu khong ghim) no se ghi de
# len moi truong Kaggle va lam VO numpy (loi "_center from numpy._core.umath").
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])

# Chi cai dung nhung gi Kaggle THUC SU thieu (pip da liet ke ro khi dung
# --no-deps). Tat ca deu la goi python thuan, khong dong vao numpy/scipy:
#   peft <0.19       ban moi hon da bo BoneConfig, ma "bone" la pp cua paper
#   hydra-zen        tang config cua peka
#   biomart          dung khi align ten gene
#   local_attention  scFoundation can
#   hestcore         HEST nap qua sys.path nen deps cua no khong duoc cai
run([sys.executable, "-m", "pip", "install", "-q",
     "peft>=0.17,<0.19", "hydra-zen", "biomart", "local_attention",
     "hestcore==1.0.3", "hf_transfer",
     "openslide-python", "mygene", "loguru", "einops-exts"])

# Kiem trong tien trinh RIENG (giong luc chay script that), khong import vao
# kernel cua notebook -> tranh dinh module da nap tu truoc.
run([sys.executable, "-c",
     "import numpy, peft, hydra_zen, biomart, local_attention; "
     "print('numpy', numpy.__version__, '| peft', peft.__version__); "
     "assert hasattr(peft, 'BoneConfig'), 'peft thieu BoneConfig'"])

(REPO / ".env").write_text(
    "HF_TOKEN=" + os.environ["HF_TOKEN"] + "\n"
    "HEST1K_STORAGE_PATH=" + str(SCRATCH) + "/HEST1K\n"
    "WANDB_API_KEY=\nWANDB_ENTITY=\n")

# --- CHAN DOAN: /kaggle/input dang co gi ------------------------------------
print("=== /kaggle/input ===")
_inp = Path("/kaggle/input")
_kids = sorted(_inp.glob("*")) if _inp.is_dir() else []
if not _kids:
    print("   RONG! Chua attach input nao.")
    print("   -> Cot phai > + Add Input > Your Work > chon notebook nay > bam +")
for _a in _kids:
    print("  ", _a.name + "/")
    for _b in sorted(_a.glob("*"))[:8]:
        print("      ", _b.name + ("/" if _b.is_dir() else ""))
        if _b.is_dir():
            for _c in sorted(_b.glob("*"))[:6]:
                print("          ", _c.name + ("/" if _c.is_dir() else ""))
print("=====================")

# --- DATA/Pretrained SONG O /kaggle/tmp, KHONG o /kaggle/working ------------
# /kaggle/working chi co quota ~20 GiB. DATA (~7 GiB) + checkpoint (~5 GiB moi
# cai, Lightning giu 2 cai: best + last) vuot quota -> chet giua chung voi
# "OSError: [Errno 28] No space left on device". /kaggle/tmp dung chung o dia
# ~70 GiB va KHONG bi quota, nen de o do roi symlink vao repo.
def stage(kind):
    real = SCRATCH / kind
    real.mkdir(parents=True, exist_ok=True)
    link = REPO / kind
    if link.is_symlink():
        link.unlink()
    elif link.is_dir():
        subprocess.run(["rsync", "-a", str(link) + "/", str(real) + "/"], check=False)
        subprocess.run(["rm", "-rf", str(link)], check=False)
    link.symlink_to(real)
    return real

def restore(kind, probe):
    real = stage(kind)
    pats = ["/".join(["*"] * d) + "/" + kind for d in range(1, 7)]
    for pat in pats:
        for cand in sorted(Path("/kaggle/input").glob(pat)):
            if not cand.is_dir():
                continue
            if probe and not (cand / probe).exists():
                continue
            print("Khoi phuc", kind, "tu", cand)
            subprocess.run(["rsync", "-a", "--ignore-existing", "--exclude", "HEST1K",
                            str(cand) + "/", str(real) + "/"], check=False)
            return True
    return False

restore("DATA", "breast/breast_in_hest/aligned_adata")
restore("Pretrained", EXP)

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO / "src")
env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# BAT BUOC dat bien moi truong THAT, khong duoc chi dua vao .env:
# peka/__init__.py nap .env SAU khi paths.py da tinh xong HEST1K_STORAGE_PATH,
# nen gia tri trong .env bi bo qua -> script 10 va 11 tro ve hai cho khac nhau.
env["HEST1K_STORAGE_PATH"] = str(SCRATCH / "HEST1K")
# Ep 1 GPU: repo khong cau hinh devices/strategy, Lightning se TU BAT DDP khi
# thay 2 GPU. Dataset dung h5py handle + fork -> DDP de deadlock.
GPU0 = dict(env)
GPU0["CUDA_VISIBLE_DEVICES"] = "0"

DATA       = REPO / "DATA"
PRETRAINED = REPO / "Pretrained"
LAST       = PRETRAINED / EXP / "last.ckpt"

# Lightning giu 2 checkpoint khac nhau va chung phuc vu 2 muc dich khac nhau:
#   last.ckpt      trang thai moi nhat  -> DUNG DE RESUME (dung optimizer/epoch)
#   best-*.ckpt    val_kd_loss thap nhat -> DUNG DE DANH GIA (PHASE 4)
# val_kd_loss quay dau sau vai epoch, nen "moi nhat" khong phai "tot nhat".
import re
def _score(q):
    m = re.search(r"val_kd_loss=([0-9.]+)", q.name)
    return float(m.group(1)) if m else float("inf")
_pool = sorted((PRETRAINED / EXP).glob("best-*.ckpt")) if (PRETRAINED / EXP).is_dir() else []
BEST = min(_pool, key=_score) if _pool else None
EMB_DIR    = DATA / "breast/breast_in_hest/peka_embed/H0/scFoundation/default_model"
print("Repo san sang. Checkpoint se o:", LAST)
print("Checkpoint tot nhat dang co :", BEST if BEST else "(chua co)")

# --- TON KHO: kiem chinh xac cai gi da khoi phuc duoc ------------------------
# Da chay nhieu version thi dung doan; de no dem va bao thang.
B = DATA / "breast/breast_in_hest"
M = B / "scLLM_embed/scFoundation/default_model"
INV = {
    "patches(.h5)":     len(list(B.glob("patches/*.h5"))),
    "aligned_adata":    len(list(B.glob("aligned_adata/*.h5ad"))),
    "paired_seq":       len(list(M.glob("paired_seq/*.h5ad"))),
    "embeddings(.npy)": len(list(M.glob("embeddings/*.npy"))),
    "peka_embed":       len(list((B / "peka_embed").rglob("*.npy"))),
    "patches_embed":    len(list((B / "patches_embed").rglob("*.npy"))),
    "last.ckpt":        len(list(PRETRAINED.rglob("last.ckpt"))),
}
print("--- TON KHO sau khoi phuc (moc chuan: 8 slide) ---")
for k, v in INV.items():
    print("   %-18s %d" % (k, v))
print("--------------------------------------------------")

# --- Checkpoint nay THUC SU o epoch bao nhieu? ------------------------------
# Ten file khong noi gi, va suy ra tu log thi chi la phong doan. Hoi thang file.
# `timer` la thoi gian Lightning cong don qua cac lan resume -- xem SessionTimer.
_cks = sorted((PRETRAINED / EXP).glob("*.ckpt")) if (PRETRAINED / EXP).is_dir() else []
if _cks:
    _p = REPO / "_ckpt_info.py"
    _p.write_text(chr(10).join([
        'import sys, torch',
        'for f in sys.argv[1:]:',
        "    c = torch.load(f, map_location='cpu', weights_only=False)",
        "    t = c.get('callbacks', {})",
        "    el = [v.get('time_elapsed', {}).get('train')",
        "          for k, v in t.items() if isinstance(v, dict) and 'time_elapsed' in v]",
        "    print('   %-14s epoch=%s  global_step=%s  timer=%s'",
        "          % (f.split('/')[-1], c.get('epoch'), c.get('global_step'),",
        "             ('%.2fh' % (el[0] / 3600)) if el and el[0] else '-'))",
    ]))
    print("--- CHECKPOINT dang co ---")
    run([sys.executable, str(_p)] + [str(c) for c in _cks], env=env)
    print("--------------------------------------------------")

# --- Lich su cac session TRUOC ----------------------------------------------
# --with_logger csv tung bi bo qua nen khong co metrics.csv, chi co tfevents.
# Doc lai de biet moi session that su chay den epoch nao -- do la cach duy nhat
# kiem chung xem resume co nhay lui hay khong.
_tb = REPO / "_read_tb.py"
_tb.write_text(chr(10).join([
        'import glob',
        'from tensorboard.backend.event_processing.event_accumulator import EventAccumulator',
        "fs = sorted(glob.glob('/kaggle/input/**/RUNS/**/events.out.tfevents.*', recursive=True))",
        'if not fs:',
        "    print('   (khong tim thay tfevents nao trong input)')",
        'for f in fs:',
        '    ea = EventAccumulator(f)',
        '    ea.Reload()',
        "    tg = ea.Tags().get('scalars', [])",
        "    ep = [x.value for x in ea.Scalars('epoch')] if 'epoch' in tg else []",
        "    vk = [x.value for x in ea.Scalars('val_kd_loss')] if 'val_kd_loss' in tg else []",
        "    run = f.split('/RUNS/')[-1].split('/')[0]",
        "    print('  ', run)",
        "    print('      epoch cao nhat :', int(max(ep)) if ep else '-')",
        "    print('      val_kd_loss    :', [round(v, 3) for v in vk])",
]))
print("--- LICH SU cac session truoc (tu tfevents) ---")
subprocess.run([sys.executable, str(_tb)], cwd=REPO, env=env, check=False)
print("--------------------------------------------------")

# --- Chot chan GPU ----------------------------------------------------------
# Script chi WARNING roi lang le chuyen sang CPU. Trich feature 30k patch qua
# ViT-g 1.1B tren CPU la nhieu NGAY, va ban chi biet khi het session.
def require_gpu():
    r = subprocess.run(
        [sys.executable, "-c", "import torch; print('GPU:', "
         "torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO')"],
        cwd=REPO, env=GPU0, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    print(out)
    assert "KHONG CO" not in out and r.returncode == 0, (
        "Session KHONG co GPU (Accelerator dang de None). Doi sang GPU T4 x2 "
        "roi chay lai -- dung CPU se mat nhieu ngay.")

# --- Tom tat PCC ------------------------------------------------------------
# Script 50 in "Mean Pearson across genes" nhung no chim giua hang nghin dong
# per-gene per-fold. Doc thang tu CSV cho chac.
def tom_tat_pcc():
    import csv
    cs = sorted(REPO.rglob("gene_regression_results.csv"), key=lambda q: q.stat().st_mtime)
    if not cs:
        print("!! Khong tim thay gene_regression_results.csv -> eval chua chay xong")
        return
    f = cs[-1]
    rows = [r for r in csv.DictReader(open(f)) if r.get("pearson_correlation")]
    if not rows:
        print("!! CSV rong:", f)
        return
    v = sorted((float(r["pearson_correlation"]), r.get("gene", "?")) for r in rows)
    mean = sum(x for x, _ in v) / len(v)
    print("=" * 60)
    print("  PCC trung binh tren %d gene : %.4f" % (len(v), mean))
    print("  H-optimus-0 dong bang       : 0.624   (moc phai vuot)")
    print("  PEKA trong paper            : 0.698")
    print("-" * 60)
    print("  te nhat :", ", ".join("%s %.3f" % (g, x) for x, g in v[:5]))
    print("  tot nhat:", ", ".join("%s %.3f" % (g, x) for x, g in v[-5:]))
    print("  CSV     :", f)
    print("=" * 60)

# --- TOAN VEN: dem file la chua du -------------------------------------------
# Lan chay chet vi het dia van sinh ra Output, nhung file trong do bi cut giua
# chung -- van dem du 8. Mo thu tung file moi biet that gia.
if INV["aligned_adata"] > 0:
    chk = REPO / "_check_integrity.py"
    chk.write_text(chr(10).join([
        'import glob, sys',
        'import numpy as np, h5py, anndata',
        "root = sys.argv[1] + '/breast/breast_in_hest'",
        "m = root + '/scLLM_embed/scFoundation/default_model'",
        'bad = []',
        "for f in sorted(glob.glob(root + '/patches/*.h5')):",
        '    try:',
        '        with h5py.File(f) as h:',
        "            _ = h['img'].shape, h['img'][0].mean()",
        '    except Exception as e:',
        '        bad.append((f, repr(e)))',
        "for f in sorted(glob.glob(root + '/aligned_adata/*.h5ad')) + sorted(glob.glob(m + '/paired_seq/*.h5ad')):",
        '    try:',
        '        _ = anndata.read_h5ad(f).shape',
        '    except Exception as e:',
        '        bad.append((f, repr(e)))',
        "for f in sorted(glob.glob(m + '/embeddings/*.npy')):",
        '    try:',
        "        a = np.load(f, mmap_mode='r')",
        '        _ = float(a[0].sum())',
        '    except Exception as e:',
        '        bad.append((f, repr(e)))',
        'for f, e in bad:',
        "    print('   HONG:', f, e)",
        'if bad:',
        "    raise SystemExit('DATA khoi phuc bi HONG/CUT (%d file). Gan nhu chac chan ban dang attach Output cua lan chay CHET GIUA CHUNG. Doi sang version chay THANH CONG.' % len(bad))",
        "print('[integrity] moi file DATA doc duoc, khong cut.')",
    ]))
    run([sys.executable, str(chk), str(DATA)], env=env)

In [ ]:
try:
    if RUN_PHASE == 1 and INV["aligned_adata"] == 8 and INV["patches(.h5)"] == 8:
        # Da co du du lieu tu dataset dinh kem -> khong lam lai.
        # (buoc 11 goi web service Ensembl BioMart, hay hong that thuong)
        print(">>> PHASE 1 DA XONG TU TRUOC (8 patches + 8 aligned_adata).")
        print(">>> Doi RUN_PHASE = 2 roi chay lai.")

    elif RUN_PHASE == 1:
        run([sys.executable, "scripts/00_check_env.py"], env=env)
        run([sys.executable, "scripts/10_download_hest1k.py",
             "--minimal", "--max-workers", "16"], env=env)
        run([sys.executable, "scripts/11_build_breast_dataset.py"], env=env)

        # --- KIEM 2 GIA DINH truoc khi dot GPU -------------------------------
        # Chay trong TIEN TRINH RIENG: kernel notebook nap numpy tu luc khoi
        # dong, sau do pip thay file ben duoi -> import anndata trong kernel vo.
        check = REPO / "_check_f1_f23.py"
        check.write_text(chr(10).join([
            'import glob, sys',
            'import h5py, anndata',
            "root = sys.argv[1] + '/breast/breast_in_hest'",
            "ph = sorted(glob.glob(root + '/patches/*.h5'))[0]",
            'f = h5py.File(ph)',
            "print('[F1] patch dtype =', f['img'].dtype, ' max =', f['img'][0].max())",
            "print('     uint8/255 => anh KHONG duoc chuan hoa truoc khi vao backbone')",
            "img_bc = [b.decode() if isinstance(b, bytes) else str(b) for b in f['barcode'][:, 0]]",
            'f.close()',
            "ad = sorted(glob.glob(root + '/aligned_adata/*.h5ad'))[0]",
            'seq = anndata.read_h5ad(ad).obs_names.to_numpy()',
            'keep = set(img_bc) & set(seq)',
            'a = [b for b in img_bc if b in keep][:200]',
            'b = [b for b in seq if b in keep][:200]',
            "print('[F23] thu tu barcode  patch == adata ?', a == b)",
            'if a != b:',
            "    raise SystemExit('[F23] THU TU BARCODE LECH. Feature se ghep NHAM nhan cua spot khac -> PCC thap gia tao. DUNG LAI, dieu tra truoc khi train.')",
            "print('     OK - feature va nhan se ghep dung.')",
        ]))
        run([sys.executable, str(check), str(DATA)], env=env)

    elif RUN_PHASE == 2:
        require_gpu()
        assert INV["aligned_adata"] == 8, (
            "Can 8 aligned_adata, chi thay %d. Version dang attach KHONG phai ban "
            "PHASE 1 hoan chinh -> doi sang version co log 'PHASE 1 xong'."
            % INV["aligned_adata"])
        run([sys.executable, "scripts/12_compute_scfoundation_emb.py"], env=GPU0)

    elif RUN_PHASE == 3:
        require_gpu()
        assert INV["embeddings(.npy)"] == 8 and INV["paired_seq"] == 8, (
            "Can 8 embedding + 8 paired_seq, chi thay %d + %d. Chay PHASE 2 truoc, "
            "hoac attach dung version da hoan thanh PHASE 2."
            % (INV["embeddings(.npy)"], INV["paired_seq"]))
        run([sys.executable, "scripts/13_kmeans_cluster_labels.py", "--use_gpu"], env=GPU0)
        run([sys.executable, "scripts/14_smoke_test_loader.py",
             "--num_batches", "2", "--num_workers", "0"], env=env)

        # --- GPU 1: DO BASELINE TRONG LUC GPU 0 TRAIN ----------------------
        # Train bi ep ve GPU 0 (DDP deadlock voi h5py+fork) nen GPU 1 nam khong
        # suot 11 tieng. Uu tien viec CHUA BIET: backbone dong bang duoi dung
        # giao thuc slide-holdout. Chua co no thi con so PEKA khong dien giai
        # duoc -- 0.624 trong paper do bang giao thuc khac.
        # Co baseline roi thi quay sang cham checkpoint moi nhat.
        EVLOG = OUT / "eval_song_song.log"
        ev = None
        CK = ((LAST if LAST.is_file() else None) or BEST) if EVAL_CKPT == "last"              else (BEST or (LAST if LAST.is_file() else None))
        E50 = "%s scripts/50_eval_gene_regression_kfold.py" % sys.executable

        # Job nen CHI trich feature. Script 50 nap ca 50 gene x 30k x 3072
        # float vao RAM cung luc (~16 GiB); chay song song voi train thi OOM
        # killer ra tay -- lan truoc no giet dung tien trinh eval o phut thu 37.
        # Phan Ridge de sau khi train xong, luc RAM da tro lai.
        if INV["patches_embed"] == 0:
            job = "BASELINE (backbone dong bang)"
            steps = ["%s scripts/42_extract_baseline_features.py --encoder %s"
                     % (sys.executable, ENCODER)]
            after = ["%s --encoder %s --feature_type image_encoder --probe_only"
                     % (E50, ENCODER)]
        elif CK:
            # Train ghi de last.ckpt moi cuoi epoch. Doc thang vao no thi co
            # luc trung dung khoanh khac file dang duoc thay -> torch.load hong.
            # Chup mot ban truoc khi train chay.
            if CK == LAST:
                SNAP = LAST.parent / "_snapshot.ckpt"
                shutil.copy2(str(LAST), str(SNAP))
                CK = SNAP
            job = "PEKA " + CK.name
            steps = [
                "%s scripts/40_extract_peka_features.py --encoder %s --peft %s"
                " --ckpt %s --output_dir %s" % (sys.executable, ENCODER, PEFT, CK, EMB_DIR),
                "%s scripts/41_compute_hvg_top50.py" % sys.executable,
            ]
            after = ["%s --encoder %s --peft %s --probe_only" % (E50, ENCODER, PEFT)]
        else:
            job, steps, after = None, None, None

        if steps:
            GPU1 = dict(env)
            GPU1["CUDA_VISIBLE_DEVICES"] = "1"
            sh = REPO / "_eval_bg.sh"
            # Doi train nap xong checkpoint (~5 GiB) roi moi nap cai thu hai.
            # Hai lan torch.load 5 GiB cung luc de day RAM host den muc OOM
            # killer ra tay -- va no se giet tien trinh to hon, tuc la train.
            sh.write_text(chr(10).join(["set -ex", "sleep 600"] + steps))
            print(">>> GPU1 chay song song:", job, "-> log o", EVLOG)
            ev = subprocess.Popen(["bash", str(sh)], cwd=REPO, env=GPU1,
                                  stdout=open(str(EVLOG), "w"),
                                  stderr=subprocess.STDOUT)
        else:
            print(">>> Chua co gi de danh gia song song")

        cmd = [sys.executable, "scripts/30_train_phase2_kd.py",
               "--encoder", ENCODER, "--peft", PEFT,
               "--epochs", str(EPOCHS),
               "--batch_size", str(BATCH_SIZE),
               "--accumulate_grad_batches", str(ACCUM),
               "--num_workers", "2",
               "--max_time", MAX_TIME,
               "--with_logger", "csv"]
        if LAST.is_file():
            print(">>> RESUME tu", LAST)
            cmd += ["--resume", str(LAST)]
        else:
            print(">>> Bat dau train tu dau")
        run(cmd, env=GPU0)

        if ev is not None:
            try:
                ev.wait(timeout=120)   # trich ~1h, train ~10h -> xong tu lau
            except subprocess.TimeoutExpired:
                ev.kill()
                print(">>> Trich feature song song qua han, da dung.")
            if ev.returncode == 0:
                # Train da nha RAM -> gio moi chay Ridge duoc an toan.
                for c in after:
                    subprocess.run(["bash", "-c", c + " >>%s 2>&1" % EVLOG],
                                   cwd=REPO, env=env, check=False)
            print("=== KET QUA SONG SONG: " + job + " ===")
            # Hai lan eval ghi de cung mot CSV, nen doc thang tu log: dong thu
            # nhat la slide-holdout (that), dong thu hai la spot-split (ro ri).
            _txt = EVLOG.read_text()
            _hit = [l.strip() for l in _txt.splitlines()
                    if "Mean Pearson across genes" in l]
            for _i, _l in enumerate(_hit):
                print("   [%s] %s" % ("slide-holdout" if _i == 0 else "spot-split ", _l))
            if not _hit:
                print(_txt[-3000:])

    elif RUN_PHASE == 4:
        require_gpu()
        if EVAL_CKPT == "best":
            CK = BEST or (LAST if LAST.is_file() else None)
        else:
            CK = (LAST if LAST.is_file() else None) or BEST
        assert CK, ("Khong thay checkpoint nao trong " + str(PRETRAINED / EXP) +
                    " -> attach version da chay PHASE 3.")
        run([sys.executable, str(REPO / "_ckpt_info.py"), str(CK)], env=env)
        # BAY: script 50 (--probe_only) tim feature trong peka_embed/H0/, nhung
        # script 40 mac dinh ghi vao peka_embed/<encoder>_<peft>/ -> ep --output_dir.
        run([sys.executable, "scripts/40_extract_peka_features.py",
             "--encoder", ENCODER, "--peft", PEFT, "--ckpt", str(CK),
             "--output_dir", str(EMB_DIR)], env=GPU0)
        run([sys.executable, "scripts/41_compute_hvg_top50.py"], env=env)
        run([sys.executable, "scripts/50_eval_gene_regression_kfold.py",
             "--encoder", ENCODER, "--peft", PEFT, "--probe_only"], env=env)
        print(">>> [1/2] Giao thuc THAT: giu nguyen slide (leave-one-slide-out)")
        tom_tat_pcc()

        # --- Doi chung: cung feature, chia ngau nhien theo spot -------------
        # Chi de CHAN DOAN. Neu con so nhay vot thi khoang cach den 0.698 la do
        # giao thuc (paper nhieu kha nang chia theo spot), khong phai do model
        # hong. Neu van ~0.1 thi feature that su co van de.
        run([sys.executable, "scripts/50_eval_gene_regression_kfold.py",
             "--encoder", ENCODER, "--peft", PEFT, "--probe_only",
             "--cv", "spot"], env=env)
        print(">>> [2/2] CHAN DOAN: chia theo spot -- RO RI, khong duoc bao cao")
        tom_tat_pcc()

    elif RUN_PHASE == 5:
        require_gpu()
        # DOI CHUNG. Backbone dong bang: khong adapter, khong translate, khong
        # checkpoint. Do duoi DUNG giao thuc slide-holdout nhu PEKA -> day moi
        # la con so PEKA phai vuot. So 0.624 trong paper do bang giao thuc nao
        # thi khong ai biet, nen khong dung de ket luan duoc.
        run([sys.executable, "scripts/42_extract_baseline_features.py",
             "--encoder", ENCODER], env=GPU0)
        run([sys.executable, "scripts/50_eval_gene_regression_kfold.py",
             "--encoder", ENCODER, "--feature_type", "image_encoder",
             "--probe_only"], env=env)
        print(">>> BASELINE [1/2] giu nguyen slide")
        tom_tat_pcc()
        run([sys.executable, "scripts/50_eval_gene_regression_kfold.py",
             "--encoder", ENCODER, "--feature_type", "image_encoder",
             "--probe_only", "--cv", "spot"], env=env)
        print(">>> BASELINE [2/2] chia theo spot -- CHAN DOAN, ro ri")
        tom_tat_pcc()
    elif RUN_PHASE == 6:
        # KHONG can GPU -- chi PCA + Ridge tren feature da co. Chay o session
        # Accelerator: None de khong ton quota GPU.
        assert INV["peka_embed"] == 8, (
            "Can 8 file peka_embed, chi thay %d. Chay PHASE 4 truoc."
            % INV["peka_embed"])
        run([sys.executable, "scripts/52_eval_official_protocol.py",
             "--encoder", ENCODER, "--peft", PEFT], env=env)

    elif RUN_PHASE == 7:
        # Chay CODE GOC cua RunningStone/PEKA tren feature cua chung ta.
        # Khong can GPU, khong can submodule. ~30-60 phut.
        assert INV["peka_embed"] == 8, (
            "Can 8 file peka_embed, chi thay %d. Chay PHASE 4 truoc."
            % INV["peka_embed"])
        _e = dict(env)
        _e["DATA_SRC"] = str(DATA)
        _e["ROOT"] = "/kaggle/tmp/official"
        _e["OUT"] = str(OUT / "official_eval")
        _e["FEATURE_TYPE"] = "peka"
        run(["bash", "scripts/53_run_official_eval.sh"], env=_e)

    elif RUN_PHASE == 8:
        # TRAIN bang chinh code cua RunningStone/PEKA. Can GPU.
        require_gpu()
        _e = dict(GPU0)
        _e["DATA_SRC"]  = str(DATA)
        _e["ROOT"]      = "/kaggle/tmp/official"
        _e["CKPT_DST"]  = "/kaggle/tmp/official/Pretrained"
        _e["OUT_DST"]   = "/kaggle/tmp/official/OUTPUT"
        _e["EPOCHS"]    = str(EPOCHS)
        _e["BATCH_SIZE"] = str(BATCH_SIZE)
        run(["bash", "scripts/54_run_official_train.sh"], env=_e)

finally:
    # Luon chay, ke ca khi phase o tren loi -> khong bao gio mat cong da lam.
    def sync(src, dst, extra=()):
        src, dst = Path(src), Path(dst)
        if not src.exists():
            return
        dst.parent.mkdir(parents=True, exist_ok=True)
        tail = "/" if src.is_dir() else ""
        subprocess.run(["rsync", "-a"] + list(extra) +
                       [str(src) + tail, str(dst) + tail], check=False)

    # Output cua session nay la Input DUY NHAT cua session sau -- Kaggle chi cho
    # attach MOT version cua cung mot dataset, khong ghep "DATA o version 2" voi
    # "ckpt o version 5" duoc. Nen moi version phai TU DU: DATA + checkpoint.
    sync(SCRATCH / "DATA", OUT / "DATA", ["--exclude", "HEST1K"])   # ~7 GiB
    sync(LAST, OUT / "Pretrained" / EXP / "last.ckpt", ["-L"])      # ~5 GiB, resume
    # Giu DUY NHAT mot best xuyen suot moi session: gom ung vien moi (epoch=*)
    # voi best cu da khoi phuc, chon cai val_kd_loss thap nhat. Khong lam viec
    # nay thi trong so tot nhat ket lai o /kaggle/tmp va mat khi het session.
    _new = list((SCRATCH / "Pretrained" / EXP).glob("epoch=*.ckpt")) + _pool
    if _new:
        _b = min(_new, key=_score)
        sync(_b, OUT / "Pretrained" / EXP / ("best-val_kd_loss=%.4f.ckpt" % _score(_b)),
             ["-L"])                                                # ~5 GiB, eval
    sync(REPO / "OUTPUT", OUT / "RUNS")                             # lora + csv, nho
    # PHASE 7/8 chay repo goc o /kaggle/tmp -- khong sync thi mat sach.
    sync("/kaggle/tmp/official/OUTPUT", OUT / "OFFICIAL_OUTPUT")
    for _c in Path("/kaggle/tmp/official/Pretrained").rglob("*.ckpt")             if Path("/kaggle/tmp/official/Pretrained").is_dir() else []:
        sync(_c, OUT / "OFFICIAL_Pretrained" / _c.name, ["-L"])

    def gib(p):
        p = Path(p)
        if not p.is_dir():
            return 0.0
        return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3
    _o = gib(OUT)
    print("OUTPUT (co quota) : %.2f GiB / ~20 GiB%s"
          % (_o, "   <-- SAT TRAN, xem lai" if _o > 18 else ""))
    print("/kaggle/tmp       : %.2f GiB (khong tinh quota)" % gib(SCRATCH))
    print("PHASE", RUN_PHASE, "-> Save Version, roi attach Output nay cho session sau.")
